In [ ]:
### CausalAttention 모듈

import torch
import torch.nn as nn

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init()
        
        ## 1. QKV layer 생성
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        ## 2. dropout layer 생성
        self.dropout = nn.Dropout(dropout)

        ## 3. mask 생성
        self.register_buffer( "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1) )

    def forward(self, x):
        b, num_tokens, d_in = x.shape   # [B, N, d_in]

        ## 1. QKV 생성
        queries = self.W_query(x)   # [B, N, d_in]  -> [B, N, d_out]
        keys = self.W_key(x)        # [B, N, d_in]  -> [B, N, d_out]
        values = self.W_value(x)    # [B, N, d_in]  -> [B, N, d_out]

        ## 2. attention score 계산
        attn_scores = queries @ keys.transpose(1, 2)    # [B, N, N]

        ## 3. attention score 에 mask 적용 (-torch.inf)
        attn_scores.maked_fill_( self.mask.bool()[:num_tokens, :num_tokens], -torch.inf )

        ## 4. attention weight 계산
        attn_weights = torch.softmax( attn_scores / keys.shape[-1]**0.5, dim=-1 )

        ## 5. attention weight dropout
        attn_weights = self.dropout(attn_weights)

        ## 5. context_vec 계산
        context_vec = attn_weights @ values     # [B, N, d_out]

        return context_vec


In [ ]:
### CausalAttention 사용해 보기

import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

d_in = inputs.shape[0]  # 3 (embeding dim)
d_out = 2               # Q, K, V 차원

batch = torch.tensor([inputs, inputs])      # B = 2, [B, N, d_in] = [2, 6, 3]

context_length = batch.shape[0]             # 6

ca = CausalAttention(d_in, d_out, context_length, 0.1)
context_vec = ca(batch)                     # [B, N, d_out] = [2, 6, 2]

In [ ]:
### MultiHeadAttention Wrapper  (concat)

class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init()

        ## heads 생성
        self.heads = nn.ModuelList(
            [ CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) for _ in range(num_heads) ]
        )

    def forward(self, x):
        return torch.cat([ head(x) for head in self.heads ], dim=-1)

In [ ]:
### MultiHeadAttentionWrapper 사용

context_length = batch.shape[1] # N=6
d_in, dout = 4, 2

mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)

context_vec = mha(batch)    # [2, 6, 4] in -> [2, 6, 2] x 2(heads) = [2, 6, 4]

In [ ]:
### MultiHeadAttention 모듈

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):    # d_out : 헤드를 다 합친 d_out
        ## 1. 상수 저장 (d_out, num_heads, d_head)
        self.d_out = d_out
        self.num_heads = num_heads
        self.d_head = d_out // num_heads

        ## 2. QKV layer 생성 (멀티 헤드 전체 크기)
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        ## 3. out projection layer 생성
        self.out_proj = nn.Linear(d_out, d_out)

        ## 4. dropout layer 생성
        self.dropout = nn.Dropout(dropout)

        ## 5. mask 생성
        self.register_buffer( "mask", torch.triu( torch.ones(context_length, context_length), diagonal=1 ) )
    
    def forward(self, x):
        ## 1. input 차원 상수 정의
        b, num_tokens, d_in = x.shape   # [B, N, d_in]

        ## 2. QKV 적용
        quries = self.W_query(x)        # [B, N, d_in]  -> [B, N, d_out] = [B, N, d_head * num_heads]
        keys = self.W_key(x)
        values = self.W_value(x)

        ## 2-1. MultiHead QKV 나누기
        quries = quries.view(b, num_tokens, self.num_heads, self.d_head)    # [B, N, d_out = num_heads * d_head] ->  #[B, N, num_heads, d_head]
        keys = keys.view(b, num_tokens, self.num_heads, self.d_head)
        values = values.view(b, num_tokens, self.num_heads, self.d_head)

        quries = quries.transpose(1, 2)     # [B, N, num_heads, d_head] -> [B, num_heads, N, d_head]
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        ## 3. attention score 계산 (QK)
        attn_scores = quries @ keys.transpose(2, 3)     # [B, num_heads, N, N]

        ## 4. attention score 에 마스크 적용
        attn_scores = attn_scores.maksed_fill_( self.mask.bool()[:num_tokens, :num_tokens], -torch.inf )

        ## 5. attention weight 계산 (QKV)
        attn_weights = torch.softmax( attn_scores / keys.shape[-1]**0.5, dim=-1)

        ## 6. dropout 적용
        attn_weights = self.dropout(attn_weights)       # [B, num_heads, N, N]

        ## 7. context_vector 계산
        context_vec = attn_weights @ values             # [B, num_heads, N, d_head]
        context_vec = context_vec.traspose(1, 2)        # -> [B, N, num_heads, d_head]

        ## 8. output project 적용
        context_vec = self.out_proj(context_vec)

        return context_vec

In [ ]:
### MultiHeadAttention 사용해 보기

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape      # [2, 6, 4]
d_out = 4

mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)     # d_head = 2

context_vec = mha(batch)        # [2, 6, 2] x 2 => [2, 6, 4]